# 01 · Send your first event and read it back
**Level:** beginner · **Time:** 30–40 minutes · **Prerequisite:** notebook 00 connects to Kafka.

You will create a topic, serialize a Python dictionary, send it, inspect its delivery report,
and consume it twice with independent groups. All loops have deadlines.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


In [ ]:
metadata = require_broker()
print('Connected to', len(metadata.brokers), 'broker(s)')


## 1. Create a fresh topic
We start with one partition so the sequence is easy to see. Topic creation uses Kafka's
Admin API. Inspect `labtools/kafka.py` later; for now focus on the event lifecycle.


In [ ]:
topic = new_topic('first-event', partitions=1)
print(topic)


## 2. Build an event
An event describes something that happened. An `event_id` identifies this occurrence;
an `order_id` identifies the business entity. They serve different purposes.


In [ ]:
from datetime import datetime, timezone
event = {
    "event_id": str(uuid4()), "event_type": "order.created", "schema_version": 1,
    "occurred_at": datetime.now(timezone.utc).isoformat(),
    "order_id": "order-1001", "customer_id": "customer-42", "total": "49.90",
}
event


## 3. Serialize to bytes
Kafka stores bytes. JSON is our initial wire format. Money is a decimal string because
binary floating point can introduce rounding errors.


In [ ]:
import json
payload = json.dumps(event, allow_nan=False).encode("utf-8")
assert json.loads(payload.decode("utf-8")) == event
print(type(payload).__name__, len(payload), "bytes")


## 4. Produce and wait for acknowledgement
`produce()` queues locally. `poll()` and `flush()` serve callbacks; `flush()` returning
zero only means the queue drained. We also inspect callback errors before claiming success.


In [ ]:
p = producer()
delivery_reports = []
def delivered(error, message):
    delivery_reports.append({"error": str(error) if error else None,
        "partition": message.partition(), "offset": message.offset()})
p.produce(topic, key=event["order_id"].encode(), value=payload, on_delivery=delivered)
p.poll(0)
remaining = p.flush(15)
assert remaining == 0, f"{remaining} records still queued"
assert len(delivery_reports) == 1 and delivery_reports[0]["error"] is None, delivery_reports
delivery_reports


## 5. Consume with the underlying API
`earliest` applies only when this group has no usable saved offset. A fresh group is
essential to this demonstration. Close the consumer in `finally`, even when an assertion fails.


In [ ]:
c = consumer(f"lab.first.{RUN}")
try:
    c.subscribe([topic])
    messages = poll_records(c, count=1, timeout=20)
    message = messages[0]
    received = decode(message)
    print("topic / partition / offset:", message.topic(), message.partition(), message.offset())
    print("key:", message.key().decode())
    assert received == event
    # The successful print/validation is our processing step for this introductory lab.
    c.commit(message=message, asynchronous=False)
finally:
    c.close()
received


## 6. Read again with an independent group
The first read did not delete the event. This time use the small `collect` helper, which
subscribes, polls with a deadline and always closes its own consumer.


In [ ]:
again = collect(topic, count=1, group_id=f"lab.audit.{RUN}")
assert decode(again[0]) == event
print("An independent group read the same event at offset", again[0].offset())


## 7. A short batch
These are three new occurrences. We reuse the topic to show that the log appends.


In [ ]:
batch = [{**event, "event_id": str(uuid4()), "order_id": f"order-{i}"} for i in range(1002, 1005)]
reports = publish(topic, batch, key_field="order_id")
print(reports)
all_messages = collect(topic, count=4)
assert [decode(m)["order_id"] for m in all_messages] == ["order-1001", "order-1002", "order-1003", "order-1004"]
assert [m.offset() for m in all_messages] == sorted(m.offset() for m in all_messages)


## Expected observation
Four events exist on one partition. The first consumer's group saved its progress;
the independent audit group still read from the beginning. A production handler must
complete its durable side effect before committing, a topic we revisit in Django.

## Exercise
Send an `order.cancelled` event for `order-1001`. Read all five events with a new group.
Which field stays the same, and which must change?
<details><summary>Solution hint</summary>
Reuse `order_id`, create a new `event_id`, change `event_type`, and publish with
`key_field='order_id'`. Collect five records using a new group. Do not assume a topic is a mutable table.
</details>

## Demonstrate aloud
Point out: dictionary → bytes → producer queue → broker acknowledgement → stored record → consumer.
Next: **02 — partitions, keys and ordering**.
